# Fraud detection on SageMaker — explore, train, evaluate

Interactive path through the same code the pipeline runs. Use this to understand the data
and tune hyperparameters; use `src/pipeline/run_pipeline.py` for anything repeatable.


In [ ]:
%pip install -q sagemaker pandas numpy scikit-learn xgboost matplotlib

In [ ]:
import boto3, sagemaker, pandas as pd, numpy as np

session = sagemaker.Session()
region = session.boto_region_name
role = sagemaker.get_execution_role()      # inside Studio; otherwise paste the stack's role ARN
bucket = "REPLACE-WITH-DataBucket-OUTPUT"

print(region, role, bucket)

## 1. Look at the imbalance before anything else

The positive class is rare enough that accuracy is a useless metric. Check the ratio, then
decide the metric (AUC-PR) and the resampling/weighting strategy.

In [ ]:
df = pd.read_csv(f"s3://{bucket}/raw/creditcard.csv")
print(df.shape)
print(df["Class"].value_counts())
print("positive rate: {:.4%}".format(df["Class"].mean()))
df.head()

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
df["Amount"].plot(kind="hist", bins=80, ax=axes[0], title="Amount (all)")
df.loc[df["Class"] == 1, "Amount"].plot(kind="hist", bins=40, ax=axes[1], title="Amount (fraud)")
plt.tight_layout()

## 2. Preprocess with the same script the pipeline uses

In [ ]:
!python ../src/pipeline/preprocess.py --input-dir ./data/raw --output-dir ./data/processed

## 3. Train locally on a sample, then on SageMaker

Local first: a 100k-row sample trains in under a minute and tells you whether the features
carry signal at all. Only then pay for a training job.

In [ ]:
!python ../src/training/train.py \
    --train ./data/processed/train \
    --validation ./data/processed/validation \
    --model-dir ./model --output-data-dir ./output --target-recall 0.85

In [ ]:
from sagemaker.xgboost.estimator import XGBoost

estimator = XGBoost(
    entry_point="train.py",
    source_dir="../src/training",
    framework_version="1.7-1",
    py_version="py3",
    role=role,
    instance_type="ml.m5.xlarge",
    instance_count=1,
    output_path=f"s3://{bucket}/models/",
    hyperparameters={"num-round": 400, "max-depth": 6, "eta": 0.1, "target-recall": 0.85},
)

estimator.fit({
    "train": f"s3://{bucket}/processed/train/",
    "validation": f"s3://{bucket}/processed/validation/",
})

## 4. Evaluate on the held-out test set

Report precision at the chosen recall, and the confusion matrix in absolute numbers —
"92% precision" means something different at 300 alerts/day than at 30,000.

In [ ]:
import json, tarfile, io, boto3
from sklearn.metrics import precision_recall_curve, average_precision_score, confusion_matrix

# metrics written by train.py, shipped inside model.tar.gz
print(json.dumps(json.load(open("./model/model_metadata.json"))["metrics"], indent=2))

## 5. Deploy

The CDK stack owns the long-lived infrastructure; deploy the approved model package to the
endpoint it declares, rather than creating ad-hoc endpoints from the notebook that nobody
remembers to delete.

In [ ]:
predictor = estimator.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name="fraud-detection-dev",
)

# remember to clean up
# predictor.delete_endpoint()